# Solutions — Lists: Slicing, Sorting, Chunking, Flattening

Exercise notebook: `02_collections/01_lists.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Paginate a result set

**Working solution**

In [ ]:
def paginate(items, page, page_size):
    if page < 1 or page_size < 1:
        raise ValueError("page and page_size must be >= 1")
    start = (page - 1) * page_size
    return items[start:start + page_size]

**Explanation**

Slicing never raises for out-of-range bounds — it just clips. That removes all the bounds checks you'd write with `subList` in Java.

**Complexity:** O(page_size).

**Common mistakes**

- Manually checking `start < len(items)` — slicing already handles it.

## Exercise 2 — 🟢 Easy / 🟡 Medium · Chunk records for batch inserts

**Working solution**

In [ ]:
def chunk(items, size):
    if size < 1:
        raise ValueError("size must be >= 1")
    return [items[i:i + size] for i in range(0, len(items), size)]

**Explanation**

`range(start, stop, step)` + slicing is the idiom. (Python 3.12 adds `itertools.batched`, which works on any iterable — you'll use it in Level 10.)

**Complexity:** O(n).

**Common mistakes**

- Appending to a `current` list and forgetting to flush the last partial batch.

## Exercise 3 — 🟡 Medium · Multi-key sorting with business priority

**Working solution**

In [ ]:
PRIORITY = {status: rank for rank, status in enumerate(["FAILED", "PENDING", "SHIPPED", "DELIVERED"])}

def sort_orders(orders):
    return sorted(orders, key=lambda o: (PRIORITY.get(o["status"], len(PRIORITY)), -o["amount"], o["order_id"]))

**Explanation**

A `key` function returning a tuple sorts lexicographically by each element — no `Comparator.comparing().thenComparing()`
chain. Negate numbers for descending inside a mixed-direction key. `sorted()` returns a new list; `list.sort()` sorts in place.

**Alternative: stable multi-pass sort (useful when a key can't be negated, e.g. strings descending)**

In [ ]:
def sort_orders_multipass(orders):
    result = sorted(orders, key=lambda o: o["order_id"])
    result.sort(key=lambda o: o["amount"], reverse=True)
    result.sort(key=lambda o: PRIORITY.get(o["status"], len(PRIORITY)))
    return result

**Complexity:** O(n log n).

**Common mistakes**

- Using `orders.sort()` and mutating the caller's data.
- Calling `list.index()` inside the key (O(k) per comparison) instead of a dict lookup.

## Exercise 4 — 🟠 Hard · Flatten arbitrarily nested lists

**Working solution**

In [ ]:
def _walk(node):
    if isinstance(node, (list, tuple)):
        for child in node:
            yield from _walk(child)
    else:
        yield node

def flatten(nested):
    return list(_walk(nested))

**Explanation**

A recursive generator with `yield from` delegates to the sub-generator — clean and lazy. Checking for
`(list, tuple)` (instead of "anything iterable") avoids the classic bug where a string is iterated into characters.

**Complexity:** O(total elements). Recursion depth = nesting depth (Python's default limit is ~1000).

**Common mistakes**

- `hasattr(x, '__iter__')` → strings explode into characters.
- Building lists at every recursion level (`result += flatten(child)`) — works but allocates a lot.

## Challenge — 🔴 Very Hard · Merge sorted event feeds from several sources

**Working solution**

In [ ]:
import heapq

def merge_feeds(feeds):
    seen = set()
    merged = []
    for event in heapq.merge(*feeds, key=lambda e: e["ts"]):
        if event["event_id"] in seen:
            continue
        seen.add(event["event_id"])
        merged.append(event)
    return merged

**Explanation**

`heapq.merge` is a lazy k-way merge: O(n log k) instead of O(n log n), and it never materializes everything at
once. It's stable across inputs, so ties come out in feed order.

**Complexity:** O(n log k) time, O(k) heap + O(unique ids) for dedup.

**Common mistakes**

- `sorted(sum(feeds, []))` — O(n²) concatenation and O(n log n) sort.
- Dedupe by comparing whole dicts instead of the business key.